# Figure 2 — analysis (young and old fibroblasts stimulated with TGF-β under tension)
Computes every Figure 2 / Supplementary Figure 2Sa / 2Sb result and writes it to `data/`; the panels are drawn from these
tables in `tgfb_figures.ipynb` and the pages assembled in `tgfb_pages.ipynb`.

Inputs (see README.md): bulk RNA-seq counts and the DESeq2 results of the TGF-β experiment (`data/`), ATAC-seq peaks
(`data/consensus_peak-8samples.bed`, per-sample peaks in `data/atac-seq/raw/`) and their DESeq2 results
(`/data/land/3d-inter-paper-data/yawen-atac-data/de-peaks/`), and the age-specific Hi-C contact graph
(`/data/land/3d-inter-paper-data/networkx_graphs/G_all.pkl`). Needs `bedtools`
installed in the same environment as Python.

In [1]:
import itertools
import json
import os
import pickle
import subprocess
import sys

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

META = "../../metadata"
DATA = "data"
G_ALL = "/data/land/3d-inter-paper-data/networkx_graphs/G_all.pkl"
DE_PEAKS_DIR = "/data/land/3d-inter-paper-data/yawen-atac-data/de-peaks/"
BEDTOOLS = os.path.join(os.path.dirname(sys.executable), "bedtools")   # bedtools installed alongside this Python

# the 60-pathway panel; the four young-colocalized pathways analysed here
pathway_genes_dict = json.load(open(f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.json"))
young_select_pathways = ['Extracellular matrix (Reactome)',
                         'TGF-beta signaling (KEGG+Reactome)',
                         'Senescence inhibitors (CellAge)',
                         'MYC targets v1 (H)']
PATHWAY_KEYS = dict(zip(young_select_pathways, ["ecm", "tgfb", "seninh", "myc"]))   # short names for file names

## Expression (Supplementary Figure 2Sa, panel a; also Figure 2c)
Bulk RNA-seq of young (Y) and old (O) fibroblasts under tension (R), control (C) or TGF-β (T), two replicates each.
Genes on chrX, chrY and chrM are dropped (as the Hi-C analysis); a gene is "active" (expressed) if its mean count over
the two replicates exceeds 100 in any of the four conditions. DESeq2 results per comparison are read from `data/deseq/`.
Writes `data/expression_tgfb.csv` (mean counts, log2FC, padj and DE label per gene).

In [2]:
gtf_df = pd.read_csv(f"{META}/hg38/hg38_gtf.csv", index_col=0)
filt_df = gtf_df[(gtf_df['feature'] == 'gene') & (gtf_df['gene_type'] == 'protein_coding')]
gene_to_chrom_dict = dict(zip(filt_df['gene_name'], filt_df['chr']))

bulk_rna_df = pd.read_csv(f"{DATA}/tgfb-force-exp-counts.csv", index_col=0)
bulk_rna_df['chr'] = bulk_rna_df['gene_name'].map(gene_to_chrom_dict)
bulk_rna_df = bulk_rna_df[~bulk_rna_df['chr'].isin(['chrM', 'chrX', 'chrY'])].copy()    # as the Hi-C analysis

CONDITIONS = ['YRT', 'YRC', 'ORT', 'ORC']
for s in CONDITIONS:
    bulk_rna_df[f'{s}_avg'] = (bulk_rna_df[f'{s}1'] + bulk_rna_df[f'{s}2']) / 2
expressed = (bulk_rna_df[[f'{s}_avg' for s in CONDITIONS]] > 100).any(axis=1)
active_genes = bulk_rna_df[expressed]['gene_name'].values
print(len(bulk_rna_df), "genes;", len(active_genes), "active")

19109 genes; 10115 active


In [3]:
def de_label(row, cond1, cond2):
    """DE category of one gene in a cond1-vs-cond2 comparison (padj < 0.05)."""
    if row['padj'] < 0.05:
        return f'Upreg {cond1}' if row['log2FoldChange'] > 0 else f'Upreg {cond2}'
    if np.isnan(row['padj']):
        return 'No/Low counts'
    return 'Not DE'


deg_df = bulk_rna_df[['gene_name']].copy()
for cond1, cond2 in zip(['YRT', 'ORT', 'ORT', 'ORC'], ['YRC', 'ORC', 'YRT', 'YRC']):
    comparison = f'{cond1}_vs_{cond2}'
    temp = pd.read_csv(f'{DATA}/deseq/{comparison}_deseq_results.csv', index_col=0)
    temp[comparison] = [de_label(row, cond1, cond2) for _, row in temp.iterrows()]
    temp = temp.loc[deg_df.index]
    deg_df[comparison] = temp[comparison]
    deg_df[f"{comparison}_lfc"] = temp['log2FoldChange']
    deg_df[f"{comparison}_padj"] = temp['padj']

expression = bulk_rna_df[['gene_name', 'chr'] + [f'{s}_avg' for s in CONDITIONS]].join(deg_df.drop(columns='gene_name'))
expression.to_csv(f"{DATA}/expression_tgfb.csv")

## Age-specific contact graph and contact classes (Figure 2b, 2c; Supplementary Figure 2Sa, panel b)
The LAS graph's loci get one bin per protein-coding gene, the bin holding its TSS (same rule as Figures 1 and 4–6).
Within a pathway's active genes, a gene forms young-specific (old-specific) contacts if its TSS bin has a young-only
(old-only) edge to the bin of another active gene of the pathway. Writes `data/pathway_gene_contacts.csv` (one row per
active pathway gene, in expression-table order) and `data/pathway_subsets.csv` (the gene lists per pathway and class, a
supplementary-table view of the same classes).

In [4]:
G_all = pickle.load(open(G_ALL, 'rb'))
# G_all's own gene lists hold every bin a gene body overlaps; replace them with the one bin holding each protein-coding
# gene's TSS (../../metadata/hg38/build_hg38_tss_annotation.py)
tss_table = pd.read_csv(f"{META}/hg38/bins_protein_coding_genes_tss_hg38_250000.csv", index_col=0)
tss_by_bin = tss_table["protein_coding_genes_tss"]          # bin -> comma-separated TSS genes (NaN if none)


def split_genes(value):
    """Genes of a TSS-table entry as a list (empty for a bin without protein-coding TSSs)."""
    return value.split(",") if isinstance(value, str) and value else []


tss_genes = {n: split_genes(tss_by_bin.get(n)) for n in G_all.nodes}
nx.set_node_attributes(G_all, tss_genes, "protein_coding_genes_with_tss")


def subset_graph_from_genes(G, geneset, node_attr_key='protein_coding_genes_with_tss'):
    """Subgraph of the loci holding any of the genes; each locus keeps its genes as 'pathway_genes_with_tss'."""
    pathway_set = set(geneset)
    node_to_pathway_genes_dict = {}
    for node, node_genes in nx.get_node_attributes(G, node_attr_key).items():
        overlap = set(node_genes) & pathway_set
        if len(overlap) > 0:
            node_to_pathway_genes_dict[node] = sorted(overlap)
    G_sub = G.subgraph(node_to_pathway_genes_dict.keys()).copy()
    nx.set_node_attributes(G_sub, node_to_pathway_genes_dict, 'pathway_genes_with_tss')
    return G_sub


def get_genes_from_graph(G, node_attr_key='pathway_genes_with_tss'):
    """Sorted genes held by the graph's loci."""
    genes = set()
    for gene_list in nx.get_node_attributes(G, node_attr_key).values():
        genes.update(gene_list)
    return sorted(genes)


def subset_edge_type(G, edge_types, induced_graph=True):
    """Subgraph of the edges whose type is in edge_types (induced on their loci by default)."""
    edges = [e for e, v in nx.get_edge_attributes(G, 'edge_type').items() if v in edge_types]
    G_subset = nx.edge_subgraph(G, edges).copy()
    if not induced_graph:
        return G_subset
    return G.subgraph(G_subset.nodes())


def get_young_and_old_genesets(G_all, pathway_genes):
    """Genes forming young-only and old-only contacts with other genes of the set."""
    G = subset_graph_from_genes(G_all, pathway_genes)
    geneset_old = get_genes_from_graph(subset_edge_type(G, 'Old', induced_graph=True))
    geneset_young = get_genes_from_graph(subset_edge_type(G, 'Young', induced_graph=True))
    return geneset_young, geneset_old

In [5]:
contact_rows = []
subset_rows = []
for pathway in young_select_pathways:
    active_pathway_genes = [g for g in pathway_genes_dict[pathway] if g in active_genes]
    geneset_young, geneset_old = get_young_and_old_genesets(G_all, active_pathway_genes)
    temp = deg_df[deg_df['gene_name'].isin(active_pathway_genes)]
    for gene_id, row in temp.iterrows():
        contact_rows.append({'pathway': pathway, 'gene_id': gene_id, 'gene_name': row.gene_name,
                             'young_contact': row.gene_name in geneset_young,
                             'old_contact': row.gene_name in geneset_old,
                             'YRT_vs_YRC_lfc': row.YRT_vs_YRC_lfc, 'ORT_vs_ORC_lfc': row.ORT_vs_ORC_lfc})
    subset_rows.append({'pathway': pathway,
                        'young_only_genes': [g for g in active_pathway_genes if g in geneset_young],
                        'other_genes': [g for g in active_pathway_genes if g not in geneset_young],
                        'n_young_only': sum(g in geneset_young for g in active_pathway_genes),
                        'n_other': sum(g not in geneset_young for g in active_pathway_genes)})
contacts = pd.DataFrame(contact_rows)
contacts.to_csv(f"{DATA}/pathway_gene_contacts.csv", index=False)
pd.DataFrame(subset_rows).to_csv(f"{DATA}/pathway_subsets.csv")
print(contacts.groupby('pathway', sort=False)[['young_contact', 'old_contact']].sum())

                                    young_contact  old_contact
pathway                                                       
Extracellular matrix (Reactome)                75           59
TGF-beta signaling (KEGG+Reactome)             48           33
Senescence inhibitors (CellAge)                55           32
MYC targets v1 (H)                             60           52


## Gene-level contact graphs (Figure 2b; Supplementary Figure 2Sa, panel b)
For each pathway, the gene-level graph of each subset — genes forming young-specific contacts | other genes | genes
forming old-specific contacts | other genes — with every gene as a node (chromosome attached) and every contact between
two of its genes' loci as an edge (edge type = contact state). Node and edge order are saved as built, so the figures
notebook redraws exactly this graph. Writes `data/gene_graphs/<pathway>_<subset>_{nodes,edges}.csv`.

In [6]:
def convert_to_gene_graph(G):
    """Gene-level graph of a locus subgraph: a gene-gene edge for every contact between their loci (first type kept)."""
    gene_labels_dict = nx.get_node_attributes(G, 'pathway_genes_with_tss')
    edge_type_dict = nx.get_edge_attributes(G, 'edge_type')
    G_gene = nx.Graph()
    added_edges = set()                                 # undirected gene pairs already added
    for (bin1, bin2), edge_type in edge_type_dict.items():
        genes1 = gene_labels_dict.get(bin1, [])
        genes2 = gene_labels_dict.get(bin2, [])
        for gene1, gene2 in itertools.product(genes1, genes2):
            if gene1 == gene2:
                continue
            edge = tuple(sorted((gene1, gene2)))
            if edge not in added_edges:
                G_gene.add_edge(gene1, gene2, edge_type=edge_type)
                added_edges.add(edge)
    # every gene of the loci keeps its chromosome, including genes without contacts
    gene_to_chrom = {}
    for node, attrs in G.nodes(data=True):
        for gene in attrs.get('pathway_genes_with_tss', []):
            gene_to_chrom.setdefault(gene, attrs.get('chrom'))
    for gene, chrom in gene_to_chrom.items():
        if gene not in G_gene:
            G_gene.add_node(gene, chrom=chrom)
    nx.set_node_attributes(G_gene, gene_to_chrom, 'chrom')
    return G_gene


def add_missing_genes(G_gene, genes_to_add, gene_to_chrom_dict):
    """Add genes whose loci have no contact in G_all as isolated nodes."""
    add = set(genes_to_add) - set(G_gene)
    missing = add - set(gene_to_chrom_dict)
    if missing:
        raise ValueError(f"Genes not in gene_to_chrom_dict: {missing}")
    for g in add:
        G_gene.add_node(g, chrom=gene_to_chrom_dict[g])
    return G_gene


def contact_subsets(pathway):
    """The pathway's four gene subsets: young-specific | other | old-specific | other (expression-table order)."""
    temp = contacts[contacts.pathway == pathway]
    return {'young_only': temp[temp.young_contact].gene_name.values,
            'young_other': temp[~temp.young_contact].gene_name.values,
            'old_only': temp[temp.old_contact].gene_name.values,
            'old_other': temp[~temp.old_contact].gene_name.values}

In [7]:
os.makedirs(f"{DATA}/gene_graphs", exist_ok=True)
for pathway in young_select_pathways:
    for subset, genes in contact_subsets(pathway).items():
        G_gene = convert_to_gene_graph(subset_graph_from_genes(G_all, genes))
        G_gene = add_missing_genes(G_gene, genes, gene_to_chrom_dict)
        stem = f"{DATA}/gene_graphs/{PATHWAY_KEYS[pathway]}_{subset}"
        pd.DataFrame({'gene': list(G_gene.nodes), 'chrom': [c for _, c in G_gene.nodes(data='chrom')]}).to_csv(
            f"{stem}_nodes.csv", index=False)
        pd.DataFrame([(u, v, t) for u, v, t in G_gene.edges(data='edge_type')],
                     columns=['gene1', 'gene2', 'edge_type']).to_csv(f"{stem}_edges.csv", index=False)
        print(f"{PATHWAY_KEYS[pathway]:6s} {subset:11s} {G_gene.number_of_nodes():4d} genes {G_gene.number_of_edges():5d} edges")

ecm    young_only    75 genes   987 edges
ecm    young_other  114 genes   152 edges
ecm    old_only      59 genes   423 edges
ecm    old_other    130 genes   366 edges
tgfb   young_only    48 genes   460 edges
tgfb   young_other   74 genes    25 edges
tgfb   old_only      33 genes   167 edges
tgfb   old_other     89 genes   110 edges
seninh young_only    55 genes   442 edges
seninh young_other   76 genes    28 edges
seninh old_only      32 genes   139 edges
seninh old_other     99 genes   191 edges


myc    young_only    60 genes   568 edges
myc    young_other  124 genes   179 edges
myc    old_only      52 genes   278 edges
myc    old_other    132 genes   230 edges


## TGF-β response by contact class (Figure 2c)
Two-sided Mann–Whitney test of the TGF-β log2FC (TGF-β vs control) between genes forming age-specific contacts and the
other active genes of the pathway: young cells by young-specific contacts, old cells by young-specific contacts, and old
cells by old-specific contacts. Writes `data/fig2c_mannwhitney.csv`.

In [8]:
KDE_TESTS = [('YRT_vs_YRC_lfc', 'young_contact'),       # row 0: young cells, young-specific contacts
             ('ORT_vs_ORC_lfc', 'young_contact'),       # row 1: old cells, young-specific contacts
             ('ORT_vs_ORC_lfc', 'old_contact')]         # row 2: old cells, old-specific contacts
test_rows = []
for pathway in young_select_pathways:
    temp = contacts[contacts.pathway == pathway]
    for row_idx, (lfc, contact) in enumerate(KDE_TESTS):
        other = temp[~temp[contact]][lfc].values
        age = temp[temp[contact]][lfc].values
        p_value = float('nan') if len(other) == 0 or len(age) == 0 else mannwhitneyu(age, other, alternative='two-sided')[1]
        test_rows.append({'pathway': pathway, 'row': row_idx, 'lfc': lfc, 'contact': contact,
                          'n_contact': len(age), 'n_other': len(other), 'p': p_value})
kde_tests = pd.DataFrame(test_rows)
kde_tests.to_csv(f"{DATA}/fig2c_mannwhitney.csv", index=False)
print(kde_tests.round(4).to_string(index=False))

                           pathway  row            lfc       contact  n_contact  n_other      p
   Extracellular matrix (Reactome)    0 YRT_vs_YRC_lfc young_contact         75      114 0.0013
   Extracellular matrix (Reactome)    1 ORT_vs_ORC_lfc young_contact         75      114 0.8694
   Extracellular matrix (Reactome)    2 ORT_vs_ORC_lfc   old_contact         59      130 0.1697
TGF-beta signaling (KEGG+Reactome)    0 YRT_vs_YRC_lfc young_contact         48       74 0.0005
TGF-beta signaling (KEGG+Reactome)    1 ORT_vs_ORC_lfc young_contact         48       74 0.3688
TGF-beta signaling (KEGG+Reactome)    2 ORT_vs_ORC_lfc   old_contact         33       89 0.6201
   Senescence inhibitors (CellAge)    0 YRT_vs_YRC_lfc young_contact         55       76 0.0002
   Senescence inhibitors (CellAge)    1 ORT_vs_ORC_lfc young_contact         55       76 0.6096
   Senescence inhibitors (CellAge)    2 ORT_vs_ORC_lfc   old_contact         32       99 0.7909
                MYC targets v1 (H)    0 

## Promoter annotation (ATAC-seq)
Promoters = TSS ± 1 kb of every protein-coding gene (GENCODE, `../../metadata/hg38/hg38_gtf.csv`), and the consensus ATAC
peaks of the 8 samples annotated with the promoters they overlap (`bedtools intersect`). Rewrites
`data/promoters_tss_1kb.bed` and `data/consensus_peak_annotated.bed`.

In [9]:
PROMOTER_DIST = 1000
pc_df = gtf_df[(gtf_df['gene_type'] == 'protein_coding') & (gtf_df['feature'] == 'gene')].copy()
pc_df["start"] = pc_df["start"].astype(int)
pc_df["end"] = pc_df["end"].astype(int)
plus = pc_df["strand"] == "+"                           # TSS = start on +, end on -
minus = pc_df["strand"] == "-"
pc_df.loc[plus, "prom_start"] = pc_df.loc[plus, "start"] - PROMOTER_DIST
pc_df.loc[plus, "prom_end"] = pc_df.loc[plus, "start"] + PROMOTER_DIST
pc_df.loc[minus, "prom_start"] = pc_df.loc[minus, "end"] - PROMOTER_DIST
pc_df.loc[minus, "prom_end"] = pc_df.loc[minus, "end"] + PROMOTER_DIST
pc_df["prom_start"] = pc_df["prom_start"].clip(lower=0).astype(int)   # BED cannot have negative coordinates
pc_df["prom_end"] = pc_df["prom_end"].astype(int)
pc_df[["chr", "prom_start", "prom_end", "gene_name"]].to_csv(f"{DATA}/promoters_tss_1kb.bed", sep="\t",
                                                             index=False, header=False)

with open(f"{DATA}/consensus_peak_annotated.bed", "w") as out:
    subprocess.run([BEDTOOLS, "intersect", "-a", f"{DATA}/consensus_peak-8samples.bed",
                    "-b", f"{DATA}/promoters_tss_1kb.bed", "-wa", "-wb"], stdout=out, check=True)
peak_promoter_df = pd.read_csv(f"{DATA}/consensus_peak_annotated.bed", sep="\t", header=None,
                               names=["peak_chr", "peak_start", "peak_end", "peak_name", "prom_chr", "prom_start",
                                      "prom_end", "gene_name"])
print(len(pc_df), "promoters;", len(peak_promoter_df), "peak-promoter overlaps")

20046 promoters; 15628 peak-promoter overlaps


## Promoter accessibility (Figure 2d)
In young fibroblasts, a gene's promoter is accessible in a condition if it overlaps a MACS2-called peak in either
replicate of that condition (Control = YRC1/2, TGF-β = YRT1/2). Share of each pathway's active genes with an accessible
promoter, by contact class. Writes `data/fig2d_promoter_accessibility.csv`.

In [10]:
promoters = pd.read_csv(f'{DATA}/promoters_tss_1kb.bed', sep='\t', header=None,
                        names=['chr', 'start', 'end', 'gene'])
pathway_gene_union = set()                              # restrict the overlap scan to the four pathways' genes
for pw in young_select_pathways:
    pathway_gene_union |= {g for g in pathway_genes_dict[pw] if g in active_genes}
prom_sub = promoters[promoters['gene'].isin(pathway_gene_union)]


def genes_accessible_in(sample_beds):
    """Genes whose promoter overlaps any peak called in the given per-sample beds."""
    peaks_by_chr = {}
    for bed in sample_beds:
        pk = pd.read_csv(bed, sep='\t', header=None, usecols=[0, 1, 2], names=['chr', 's', 'e'])
        for chrom, grp in pk.groupby('chr'):
            peaks_by_chr.setdefault(chrom, []).append(grp[['s', 'e']].to_numpy())
    peaks_by_chr = {chrom: np.vstack(v) for chrom, v in peaks_by_chr.items()}
    accessible = set()
    for chrom, start, end, gene in prom_sub.itertuples(index=False):
        arr = peaks_by_chr.get(chrom)
        if arr is None:
            continue
        if np.any((arr[:, 0] < end) & (arr[:, 1] > start)):    # peak.start < promoter.end and peak.end > promoter.start
            accessible.add(gene)
    return accessible

In [11]:
RAW = f'{DATA}/atac-seq/raw'
accessible_by_cond = {'Control': genes_accessible_in([f'{RAW}/YRC1_peaks.bed', f'{RAW}/YRC2_peaks.bed']),
                      'TGF-β': genes_accessible_in([f'{RAW}/YRT1_peaks.bed', f'{RAW}/YRT2_peaks.bed'])}

access_rows = []
for pathway in young_select_pathways:
    temp = contacts[contacts.pathway == pathway].drop_duplicates('gene_name')    # one entry per gene (no-op here)
    groups = {'Young-specific': temp[temp.young_contact].gene_name.tolist(),
              'Other': temp[~temp.young_contact].gene_name.tolist()}
    for cls, genes in groups.items():
        for cond, accessible in accessible_by_cond.items():
            access_rows.append({'pathway': pathway, 'contact': cls, 'condition': cond, 'n_genes': len(genes),
                                'n_accessible': sum(g in accessible for g in genes)})
accessibility = pd.DataFrame(access_rows)
accessibility.to_csv(f"{DATA}/fig2d_promoter_accessibility.csv", index=False)
print(accessibility.to_string(index=False))

                           pathway        contact condition  n_genes  n_accessible
   Extracellular matrix (Reactome) Young-specific   Control       75            65
   Extracellular matrix (Reactome) Young-specific     TGF-β       75            69
   Extracellular matrix (Reactome)          Other   Control      114            91
   Extracellular matrix (Reactome)          Other     TGF-β      114            96
TGF-beta signaling (KEGG+Reactome) Young-specific   Control       48            45
TGF-beta signaling (KEGG+Reactome) Young-specific     TGF-β       48            46
TGF-beta signaling (KEGG+Reactome)          Other   Control       74            65
TGF-beta signaling (KEGG+Reactome)          Other     TGF-β       74            65
   Senescence inhibitors (CellAge) Young-specific   Control       55            54
   Senescence inhibitors (CellAge) Young-specific     TGF-β       55            54
   Senescence inhibitors (CellAge)          Other   Control       76            70
   S

## Promoter peak changes under TGF-β (Supplementary Figure 2Sa, panel c)
DESeq2 log2FC (TGF-β vs control) of the consensus peaks overlapping each gene's promoter, averaged per gene, for the
active genes of each pathway with a promoter peak; Mann–Whitney test of young-specific vs other genes per pathway and age,
BH-adjusted over the 8 tests. Writes `data/fig2Sa_c_promoter_peak_de.csv` (one row per gene) and
`data/fig2Sa_c_promoter_peak_de_stats.csv`.

In [12]:
def build_gene_accessibility(de_peaks_df, peak_promoter_df):
    """Gene-level promoter-peak change: mean log2FC and min padj over the peaks overlapping the gene's promoter."""
    peak_to_gene = peak_promoter_df[["peak_name", "gene_name"]]
    peak_gene_df = peak_to_gene.merge(de_peaks_df[["log2FoldChange", "padj"]], left_on="peak_name", right_index=True,
                                      how="inner")
    return peak_gene_df.groupby("gene_name").agg(mean_log2fc=("log2FoldChange", "mean"), min_padj=("padj", "min"),
                                                 n_peaks=("peak_name", "nunique"))


gene_peak_change = {
    'young': build_gene_accessibility(pd.read_csv(f'{DE_PEAKS_DIR}YRT_vs_YRC_deseq_peaks_results.csv', index_col=0),
                                      peak_promoter_df),
    'old': build_gene_accessibility(pd.read_csv(f'{DE_PEAKS_DIR}ORT_vs_ORC_deseq_peaks_results.csv', index_col=0),
                                    peak_promoter_df)}

peak_rows = []
for pathway in young_select_pathways:
    active_pathway_genes = [g for g in pathway_genes_dict[pathway] if g in active_genes]
    in_pathway = contacts[contacts.pathway == pathway]
    geneset_young = set(in_pathway[in_pathway.young_contact].gene_name)        # contact classes from above
    for age in ['young', 'old']:
        gene_df = gene_peak_change[age]
        for gene in active_pathway_genes:
            if gene not in gene_df.index:
                continue                                # no promoter peak for this gene
            peak_rows.append({'pathway': pathway, 'age': age, 'gene': gene,
                              'contact': 'Young-specific' if gene in geneset_young else 'Other',
                              'mean_log2fc': gene_df.at[gene, 'mean_log2fc'], 'min_padj': gene_df.at[gene, 'min_padj']})
records_df = pd.DataFrame(peak_rows)
records_df.to_csv(f"{DATA}/fig2Sa_c_promoter_peak_de.csv", index=False)

In [13]:
stats_rows = []
for pathway in young_select_pathways:
    for age in ['young', 'old']:
        sub = records_df[(records_df['pathway'] == pathway) & (records_df['age'] == age)]
        young_lfc = sub.loc[sub['contact'] == 'Young-specific', 'mean_log2fc'].dropna()
        other_lfc = sub.loc[sub['contact'] == 'Other', 'mean_log2fc'].dropna()
        pval = np.nan
        if len(young_lfc) > 0 and len(other_lfc) > 0:
            pval = mannwhitneyu(young_lfc, other_lfc, alternative='two-sided').pvalue
        stats_rows.append({'pathway': pathway, 'age': age, 'n_young': len(young_lfc), 'n_other': len(other_lfc),
                           'mean_lfc_young': young_lfc.mean(), 'mean_lfc_other': other_lfc.mean(), 'pval': pval})
stats_df = pd.DataFrame(stats_rows)
stats_df['fdr'] = multipletests(stats_df['pval'], method='fdr_bh')[1]     # BH over all pathway/age tests
stats_df.to_csv(f"{DATA}/fig2Sa_c_promoter_peak_de_stats.csv", index=False)
print(stats_df.round(3).to_string(index=False))

                           pathway   age  n_young  n_other  mean_lfc_young  mean_lfc_other  pval   fdr
   Extracellular matrix (Reactome) young       69       94          -0.108           0.034 0.002 0.020
   Extracellular matrix (Reactome)   old       69       94           0.126           0.091 0.761 0.903
TGF-beta signaling (KEGG+Reactome) young       46       65          -0.042          -0.067 0.328 0.726
TGF-beta signaling (KEGG+Reactome)   old       46       65           0.115           0.060 0.397 0.726
   Senescence inhibitors (CellAge) young       54       71          -0.039          -0.029 0.863 0.903
   Senescence inhibitors (CellAge)   old       54       71           0.070           0.078 0.903 0.903
                MYC targets v1 (H) young       57      119           0.030          -0.004 0.110 0.439
                MYC targets v1 (H)   old       57      119           0.148           0.120 0.454 0.726


## Locus gene density and contact groups (Supplementary Figure 2Sb)
Active pathway genes in three groups: loci with no detected interchromosomal contact (TSS bin absent from the graph),
other contacts only, and young-specific contacts; locus gene density = protein-coding TSSs in the gene's 250 kb bin.
Two-sided Mann–Whitney tests of young-specific vs other-contact genes (gene density; TGF-β log2FC in young and old cells),
per pathway, descriptive. Writes `data/fig2Sb_gene_groups.csv` and `data/fig2Sb_mannwhitney.csv`.

In [14]:
gene_bin = {}                                           # gene -> its TSS bin
for b, genes in tss_genes.items():
    for g in genes:
        gene_bin.setdefault(g, b)
for b, value in tss_by_bin.items():                     # genes whose TSS bin is not a graph node
    for g in split_genes(value):
        gene_bin.setdefault(g, b)
genes_per_bin = tss_by_bin.map(lambda value: len(split_genes(value)))

groups = contacts.copy()
groups['bin'] = groups.gene_name.map(gene_bin)
groups['in_graph'] = groups['bin'].isin(set(G_all.nodes))
groups['genes_in_locus'] = groups['bin'].map(genes_per_bin)
groups['group'] = np.select([~groups.in_graph, groups.young_contact], ['no_contact', 'young_contact'], 'other_contact')
groups.to_csv(f"{DATA}/fig2Sb_gene_groups.csv", index=False)
print(groups.groupby(['pathway', 'group'], sort=False).size().unstack())

group                               no_contact  other_contact  young_contact
pathway                                                                     
Extracellular matrix (Reactome)             50             64             75
TGF-beta signaling (KEGG+Reactome)          38             36             48
Senescence inhibitors (CellAge)             35             41             55
MYC targets v1 (H)                          55             69             60


In [15]:
density_tests = []
for pathway in young_select_pathways:
    sub = groups[groups.pathway == pathway]
    young = sub[sub.group == 'young_contact']
    other = sub[sub.group == 'other_contact']
    for value in ['genes_in_locus', 'YRT_vs_YRC_lfc', 'ORT_vs_ORC_lfc']:
        p = mannwhitneyu(young[value], other[value], alternative='two-sided').pvalue
        density_tests.append({'pathway': pathway, 'value': value, 'n_young_contact': len(young),
                              'n_other_contact': len(other), 'p': p})
density_tests = pd.DataFrame(density_tests)
density_tests.to_csv(f"{DATA}/fig2Sb_mannwhitney.csv", index=False)
print(density_tests.round(4).to_string(index=False))

                           pathway          value  n_young_contact  n_other_contact      p
   Extracellular matrix (Reactome) genes_in_locus               75               64 0.0000
   Extracellular matrix (Reactome) YRT_vs_YRC_lfc               75               64 0.1015
   Extracellular matrix (Reactome) ORT_vs_ORC_lfc               75               64 0.2655
TGF-beta signaling (KEGG+Reactome) genes_in_locus               48               36 0.0038
TGF-beta signaling (KEGG+Reactome) YRT_vs_YRC_lfc               48               36 0.0181
TGF-beta signaling (KEGG+Reactome) ORT_vs_ORC_lfc               48               36 0.6416
   Senescence inhibitors (CellAge) genes_in_locus               55               41 0.0656
   Senescence inhibitors (CellAge) YRT_vs_YRC_lfc               55               41 0.0523
   Senescence inhibitors (CellAge) ORT_vs_ORC_lfc               55               41 0.8998
                MYC targets v1 (H) genes_in_locus               60               69 0.0596